In [2]:
%load_ext autoreload
%autoreload 2

import fastf1
from src.cleaning import clean_laps, add_gap_ahead

fastf1.Cache.configure(cache_dir='cache')
session = fastf1.get_session(2023, 'Bahrain', 'R')
session.load(telemetry = False, weather = False, messages = False)
laps = session.laps



core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.9.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '14', '55', '44', '18', '63', '77', '10', '23', '22', '2', '20', '21', '27', '24', '4', '31', '16', '81']


In [3]:
cleaned, funnel = clean_laps(laps)
print('Laps at the start:', len(laps), 'Laps left:', len(cleaned))
funnel

Laps at the start: 1056 Laps left: 889


,Step,Laps left,Laps removed
0,Start,1056,0
1,Remove in laps and out laps,954,102
2,Remove lap 1,934,20
3,Remove inaccurate laps,914,20
4,Remove laps with no lap time,914,0
5,Keep only green flag laps,889,25


In [4]:
print(cleaned[['Driver', 'LapNumber', 'LapTime', 'LapTimeSeconds']].head())
print('Fastest lap:', cleaned['LapTimeSeconds'].min())

  Driver  LapNumber                LapTime  LapTimeSeconds
2    VER        3.0 0 days 00:01:38.006000          98.006
3    VER        4.0 0 days 00:01:37.976000          97.976
4    VER        5.0 0 days 00:01:38.035000          98.035
5    VER        6.0 0 days 00:01:37.986000          97.986
6    VER        7.0 0 days 00:01:38.021000          98.021
Fastest lap: 93.996


In [5]:
with_gaps = add_gap_ahead(laps, cleaned)
print('Laps:', len(with_gaps))
print('Laps with a gap:', with_gaps['GapAhead'].notna().sum(), 'No car ahead:', with_gaps['GapAhead'].isna().sum())
print('Laps in traffic:', with_gaps['InTraffic'].sum())

Laps: 889
Laps with a gap: 840 No car ahead: 49
Laps in traffic: 134


In [6]:
ver = with_gaps[with_gaps['Driver'] == 'VER']
print('Verstappen laps left:', len(ver))
print(ver.groupby('Compound')['LapTimeSeconds'].agg(['count', 'min', 'max']))
print('Missing laps:', sorted(set(range(1, int(laps['LapNumber'].max()) + 1)) - set(ver['LapNumber'])))

Verstappen laps left: 49
          count     min     max
Compound                       
HARD         18  96.236  97.427
SOFT         31  97.372  98.591
Missing laps: [1, 2, 14, 15, 36, 37, 41, 42]


In [7]:
with_gaps[(with_gaps['Driver'] == 'SAI') & (with_gaps['LapNumber'].between(43, 47))][['LapNumber', 'LapTimeSeconds', 'DriverAhead', 'GapAhead', 'InTraffic']]

,LapNumber,LapTimeSeconds,DriverAhead,GapAhead,InTraffic
766,43.0,97.294,PER,23.729,False
767,44.0,97.389,PER,24.140,False
768,45.0,99.561,ALO,0.988,True
769,46.0,97.844,ALO,1.745,False
770,47.0,98.222,ALO,2.641,False
